In [ ]:
# Cell 0: Imports
import os
import sys
import json
import time
import random
import hashlib
import platform
import struct
import subprocess
import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
# Cell 1: Configuration
AUG0 = {
    "reflect_pad": 4,
    "crop_size": 32,
    "hflip": True,
    "mix": "none",
    "cutout_size": 16,
    "mixup_alpha": 0.4,
    "cutmix_alpha": 1.0,
}
TRAIN0 = {
    "momentum": 0.9,
    "eval_batch_size": 256,
    "print_every": 10,
    "flip_avg": True,
    "dropout": 0.0,
}
CONFIG = {
    "seeds": {"split": 42, "train": (0, 1, 2)},
    "data": {
        "train_images": "dataset/train-images-idx3-ubyte",
        "train_labels": "dataset/train-labels-idx1-ubyte",
        "labels_file": "dataset/labels.txt",
        "num_train": 1979,
        "image_shape": (32, 32, 3),
        "num_classes": 10,
        "expected_class_counts": (217, 216, 195, 220, 204, 204, 178, 180, 174, 191),
    },
    "paths": {"results_dir": "results"},
    "augmentation": dict(AUG0),
    "train": dict(TRAIN0),
    "expected_params": {"resnet_32_64_128": 308650},
    "defaults": {
        "model": "resnet_32_64_128",
        "lr": 0.1,
        "wd": 2e-3,
        "epochs": 480,
        "batch_size": 32,
        "seeds": (0, 1, 2),
    },
    "experiments": {
        "DROP_EP480": {"train": {"dropout": 0.2}},
        "DROP_EP480_MIX": {"aug": {"mix": "mixup"}, "train": {"dropout": 0.2}},
        "DROP_EP480_CUTOUT": {"aug": {"mix": "cutout"}, "train": {"dropout": 0.2}},
        "DROP_EP480_CUTMIX": {"aug": {"mix": "cutmix"}, "train": {"dropout": 0.2}},
    },
    "final_check": {"parity_max_mismatch_frac": 0.005},
    "runtime": {
        "deterministic": True,
        "cudnn_benchmark": False,
        "cudnn_allow_tf32": False,
        "cublas_workspace_config": ":4096:8",
        "nvidia_smi_timeout_s": 10,
    },
}

print(json.dumps(CONFIG, indent=2))

In [ ]:
# Cell 2: Environment record and seeding
RT = CONFIG["runtime"]

if torch.cuda.is_initialized() and os.environ.get("CUBLAS_WORKSPACE_CONFIG") != RT["cublas_workspace_config"]:
    raise RuntimeError("CUDA was initialised before CUBLAS_WORKSPACE_CONFIG was set; restart the kernel and run Cells 0-2 first")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = RT["cublas_workspace_config"]

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available; refusing to fall back to CPU. Check the GPU driver, the PyTorch CUDA build and WSL2 GPU access")

DEVICE = torch.device("cuda:0")
capability = torch.cuda.get_device_capability(0)
arch_needed = f"sm_{capability[0]}{capability[1]}"
arch_list = torch.cuda.get_arch_list()
if arch_needed not in arch_list:
    raise RuntimeError(f"Installed PyTorch build lists {arch_list} but the GPU needs {arch_needed}; install a build with CUDA 12.8 or newer")

torch.use_deterministic_algorithms(RT["deterministic"])
torch.backends.cudnn.deterministic = RT["deterministic"]
torch.backends.cudnn.benchmark = RT["cudnn_benchmark"]
torch.backends.cudnn.allow_tf32 = RT["cudnn_allow_tf32"]


def smoke_test():
    x = torch.randn(4, 3, 8, 8, device=DEVICE, requires_grad=True)
    conv = torch.nn.Conv2d(3, 4, 3, padding=1).to(DEVICE)
    out = (conv(x).mean(dim=(2, 3)) @ torch.ones(4, 2, device=DEVICE)).sum()
    out.backward()
    torch.cuda.synchronize()
    return bool(torch.isfinite(x.grad).all().item())


if not smoke_test():
    raise RuntimeError("CUDA conv/matmul forward-backward smoke test produced non-finite gradients")

try:
    driver = subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=RT["nvidia_smi_timeout_s"], check=True,
    ).stdout.strip()
except (OSError, subprocess.SubprocessError) as exc:
    driver = f"unavailable ({type(exc).__name__})"

props = torch.cuda.get_device_properties(0)
ENV_RECORD = {
    "timestamp": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "venv": sys.prefix,
    "numpy": np.__version__,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "driver": driver,
    "gpu": props.name,
    "gpu_capability": f"{capability[0]}.{capability[1]}",
    "gpu_memory_gb": round(props.total_memory / 1024**3, 2),
    "arch_list": arch_list,
    "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
    "float32_matmul_precision": torch.get_float32_matmul_precision(),
    "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
    "seeds": CONFIG["seeds"],
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seeds"]["split"])

print(json.dumps(ENV_RECORD, indent=2, default=str))
print(f"Cell 2 done: device={DEVICE}, seed={CONFIG['seeds']['split']}")

In [ ]:
# Cell 3: Data loading and integrity checks
D = CONFIG["data"]
ROWS, COLS, CH = D["image_shape"]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_dog_mnist(images_path, labels_path):
    with open(labels_path, "rb") as f:
        magic, n_labels = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    if magic != 2049 or labels.size != n_labels:
        raise ValueError(f"Bad label file: magic={magic}, header count={n_labels}, bytes={labels.size}")
    with open(images_path, "rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        pixels = np.frombuffer(f.read(), dtype=np.uint8)
    if magic != 2051 or n != n_labels or (rows, cols) != (ROWS, COLS) or pixels.size != n * ROWS * COLS * CH:
        raise ValueError(f"Bad image file: magic={magic}, n={n}, rows={rows}, cols={cols}, bytes={pixels.size}, labels={n_labels}")
    return pixels.reshape(n, ROWS, COLS, CH).copy(), labels.astype(np.int64)


X_TRAIN, Y_TRAIN = load_dog_mnist(D["train_images"], D["train_labels"])

if X_TRAIN.shape != (D["num_train"], ROWS, COLS, CH) or X_TRAIN.dtype != np.uint8:
    raise ValueError(f"Unexpected image array: shape={X_TRAIN.shape}, dtype={X_TRAIN.dtype}")
if Y_TRAIN.min() < 0 or Y_TRAIN.max() >= D["num_classes"]:
    raise ValueError(f"Label out of range: min={Y_TRAIN.min()}, max={Y_TRAIN.max()}")

class_counts = np.bincount(Y_TRAIN, minlength=D["num_classes"])
if class_counts.tolist() != list(D["expected_class_counts"]):
    raise ValueError(f"Class counts differ from README: got {class_counts.tolist()}, expected {list(D['expected_class_counts'])}")

image_std = X_TRAIN.reshape(len(X_TRAIN), -1).astype(np.float32).std(axis=1)
if image_std.min() <= 0:
    raise ValueError(f"Constant image found at indices {np.flatnonzero(image_std <= 0).tolist()}")

by_bytes = {}
for i in range(len(X_TRAIN)):
    by_bytes.setdefault(X_TRAIN[i].tobytes(), []).append(i)
EXACT_DUP_GROUPS = [g for g in by_bytes.values() if len(g) > 1]
dup_label_conflicts = [g for g in EXACT_DUP_GROUPS if len(set(Y_TRAIN[g].tolist())) > 1]
if dup_label_conflicts:
    raise ValueError(f"Exact duplicates with conflicting labels: {dup_label_conflicts}")

with open(D["labels_file"], encoding="utf-8") as f:
    LABELS_RAW = [line.rstrip("\n") for line in f if line.strip()]

DATA_RECORD = {
    "train_images_sha256": sha256_file(D["train_images"]),
    "train_labels_sha256": sha256_file(D["train_labels"]),
    "num_images": int(X_TRAIN.shape[0]),
    "class_counts": class_counts.tolist(),
    "pixel_min": int(X_TRAIN.min()),
    "pixel_max": int(X_TRAIN.max()),
    "min_image_std": round(float(image_std.min()), 3),
    "exact_duplicate_groups": len(EXACT_DUP_GROUPS),
    "exact_duplicate_images": sum(len(g) for g in EXACT_DUP_GROUPS),
}

print(json.dumps(DATA_RECORD, indent=2))
print("exact duplicate groups:", EXACT_DUP_GROUPS)
print("labels file:", LABELS_RAW)
print(f"Cell 3 done: X_TRAIN={X_TRAIN.shape} {X_TRAIN.dtype}, Y_TRAIN={Y_TRAIN.shape} {Y_TRAIN.dtype}")

In [ ]:
# Cell 4: Data tensors, normalization and augmentation
AUG = CONFIG["augmentation"]
PAD, SIZE = AUG["reflect_pad"], AUG["crop_size"]
MIX = AUG["mix"]
CUTOUT = AUG["cutout_size"]
NC = CONFIG["data"]["num_classes"]
if MIX not in ("none", "cutout", "cutmix", "mixup"):
    raise ValueError(f"Unknown augmentation mix mode: {MIX}")
if not 1 <= CUTOUT <= SIZE:
    raise ValueError(f"cutout_size {CUTOUT} must be between 1 and {SIZE}")
X_ALL = torch.from_numpy(X_TRAIN).permute(0, 3, 1, 2).contiguous().to(DEVICE)
Y_ALL = torch.from_numpy(Y_TRAIN).to(DEVICE)


def norm_stats(idx):
    x = X_ALL[torch.as_tensor(idx, device=DEVICE)].float() / 255.0
    return x.mean(dim=(0, 2, 3)), x.std(dim=(0, 2, 3), correction=0)


def normalize(idx, mean, std):
    x = X_ALL[torch.as_tensor(idx, device=DEVICE)].float() / 255.0
    return (x - mean[None, :, None, None]) / std[None, :, None, None]


def epoch_plan(n, gen):
    perm = torch.randperm(n, generator=gen)
    oy = torch.randint(0, 2 * PAD + 1, (n,), generator=gen)
    ox = torch.randint(0, 2 * PAD + 1, (n,), generator=gen)
    flip = (torch.rand(n, generator=gen) < 0.5) & bool(AUG["hflip"])
    return perm.to(DEVICE), oy.to(DEVICE), ox.to(DEVICE), flip.to(DEVICE)


@torch.no_grad()
def augment(xb, oy, ox, flip):
    b, c = xb.shape[0], xb.shape[1]
    padded = F.pad(xb, (PAD, PAD, PAD, PAD), mode="reflect")
    ar = torch.arange(SIZE, device=xb.device)
    rows = (oy[:, None] + ar[None, :])[:, None, :, None]
    cols = (ox[:, None] + ar[None, :])[:, None, None, :]
    bi = torch.arange(b, device=xb.device)[:, None, None, None]
    ci = torch.arange(c, device=xb.device)[None, :, None, None]
    out = padded[bi, ci, rows, cols]
    return torch.where(flip[:, None, None, None], out.flip(3), out)


def make_mix_rng(seed):
    return np.random.default_rng(seed + 1000)


def box_mask(y0, x0, h, w):
    ar = torch.arange(SIZE, device=DEVICE)
    rm = (ar[None, :] >= y0[:, None]) & (ar[None, :] < (y0 + h)[:, None])
    cm = (ar[None, :] >= x0[:, None]) & (ar[None, :] < (x0 + w)[:, None])
    return (rm[:, :, None] & cm[:, None, :])[:, None]


@torch.no_grad()
def mix_batch(xb, yb, rng, method):
    if method == "none":
        return xb, None, 1.0
    b = xb.shape[0]
    if method == "cutout":
        y0 = torch.as_tensor(rng.integers(SIZE, size=b) - CUTOUT // 2, device=DEVICE)
        x0 = torch.as_tensor(rng.integers(SIZE, size=b) - CUTOUT // 2, device=DEVICE)
        return xb.masked_fill(box_mask(y0, x0, CUTOUT, CUTOUT), 0.0), None, 1.0
    partner_x, partner_y = xb.flip(0), yb.flip(0)
    if method == "mixup":
        lam = float(rng.beta(AUG["mixup_alpha"], AUG["mixup_alpha"]))
        return lam * xb + (1.0 - lam) * partner_x, partner_y, lam
    lam0 = float(rng.beta(AUG["cutmix_alpha"], AUG["cutmix_alpha"]))
    side = int(SIZE * float(np.sqrt(1.0 - lam0)))
    cy, cx = int(rng.integers(SIZE)), int(rng.integers(SIZE))
    ys, xs = cy - side // 2, cx - side // 2
    y1, y2 = max(ys, 0), min(ys + side, SIZE)
    x1, x2 = max(xs, 0), min(xs + side, SIZE)
    out = xb.clone()
    out[:, :, y1:y2, x1:x2] = partner_x[:, :, y1:y2, x1:x2]
    lam = 1.0 - (y2 - y1) * (x2 - x1) / (SIZE * SIZE)
    return out, partner_y, lam


_t = torch.arange(2 * 3 * SIZE * SIZE, dtype=torch.float32, device=DEVICE).reshape(2, 3, SIZE, SIZE)
_mid = torch.full((2,), PAD, device=DEVICE)
_zero = torch.zeros(2, dtype=torch.long, device=DEVICE)
_no, _yes = torch.zeros(2, dtype=torch.bool, device=DEVICE), torch.ones(2, dtype=torch.bool, device=DEVICE)
assert torch.equal(augment(_t, _mid, _mid, _no), _t)
assert torch.equal(augment(_t, _mid, _mid, _yes), _t.flip(3))
assert torch.equal(augment(_t, _zero, _zero, _no)[:, :, PAD:, PAD:], _t[:, :, :SIZE - PAD, :SIZE - PAD])

_rng = make_mix_rng(0)
_xb = torch.randn(8, 3, SIZE, SIZE, device=DEVICE)
_yb = torch.arange(8, device=DEVICE)
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "none")
assert _x is _xb and _y2 is None and _lam == 1.0
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "cutout")
assert _y2 is None and _lam == 1.0 and (_x == 0).any()
assert not ((_x != _xb) & (_x != 0)).any()
assert ((_x != _xb).float().mean(dim=(1, 2, 3)) <= CUTOUT * CUTOUT / (SIZE * SIZE) + 1e-6).all()
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "mixup")
assert 0.0 <= _lam <= 1.0 and torch.equal(_y2, _yb.flip(0))
assert torch.allclose(_x, _lam * _xb + (1.0 - _lam) * _xb.flip(0), atol=1e-6)
_x, _y2, _lam = mix_batch(_xb, _yb, _rng, "cutmix")
assert 0.0 <= _lam <= 1.0 and torch.equal(_y2, _yb.flip(0))
assert abs((_x != _xb).any(dim=1).float().mean().item() - (1.0 - _lam)) < 1e-5
assert make_mix_rng(0).integers(1000) == make_mix_rng(0).integers(1000)

In [ ]:
# Cell 5: Model definitions and parameter-count check
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.short = None
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + (x if self.short is None else self.short(x)))


class ResNet(nn.Module):
    def __init__(self, widths, num_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        blocks, cin = [], widths[0]
        for s, w in enumerate(widths):
            blocks.append(Block(cin, w, 1 if s == 0 else 2))
            cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        return self.fc(self.blocks(self.stem(x)).mean(dim=(2, 3)))


def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)


MODEL_BUILDERS = {
    "resnet_32_64_128": lambda nc: ResNet((32, 64, 128), nc),
}


def build_model(key):
    m = MODEL_BUILDERS[key](NC)
    m.apply(init_weights)
    return m.to(DEVICE)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


m = build_model("resnet_32_64_128").eval()
with torch.no_grad():
    out_shape = tuple(m(torch.zeros(2, 3, SIZE, SIZE, device=DEVICE)).shape)
got, want = count_params(m), CONFIG["expected_params"]["resnet_32_64_128"]
print(f"resnet_32_64_128 params={got} expected={want} out={out_shape}")
assert got == want, f"parameter count {got} != expected {want}"
assert out_shape == (2, NC)
del m

In [ ]:
# Cell 6: Training and validation
T = CONFIG["train"]


@torch.no_grad()
def predict_logits(model, x, batch_size):
    model.eval()
    do_flip = T.get("flip_avg", True)
    preds = []
    for start in range(0, x.shape[0], batch_size):
        end = min(start + batch_size, x.shape[0])
        xb = x[start:end]
        logits = model(xb)
        if do_flip:
            logits = (logits + model(xb.flip(3))) / 2
        preds.append(logits)
    return torch.cat(preds)


def make_optimizer(model, lr, wd):
    params = list(model.parameters())
    groups = [
        {"params": [p for p in params if p.ndim > 1], "weight_decay": wd},
        {"params": [p for p in params if p.ndim <= 1], "weight_decay": 0.0},
    ]
    return torch.optim.SGD(groups, lr=lr, momentum=T["momentum"])


def make_scheduler(opt, epochs, steps):
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * steps)


def attach_dropout(model):
    p = float(T["dropout"])
    if p > 0.0:
        model.fc.register_forward_pre_hook(lambda mod, inp: (F.dropout(inp[0], p, mod.training),))
    return model


def train_one(model_key, lr, wd, epochs, batch_size, seed, train_idx, val_idx, prefix=""):
    set_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    mix_rng = make_mix_rng(seed)
    model = attach_dropout(build_model(model_key))
    mean, std = norm_stats(train_idx)
    x_tr = normalize(train_idx, mean, std)
    y_tr = Y_ALL[torch.as_tensor(train_idx, device=DEVICE)]
    if val_idx is not None:
        x_va = normalize(val_idx, mean, std)
        y_va = Y_ALL[torch.as_tensor(val_idx, device=DEVICE)]
    n = x_tr.shape[0]
    steps = n // batch_size
    opt = make_optimizer(model, lr, wd)
    sched = make_scheduler(opt, epochs, steps)
    history, logits, t0 = [], None, time.time()
    bar = tqdm(range(epochs), desc=prefix.strip() or "train", leave=False, dynamic_ncols=True)
    for epoch in bar:
        model.train()
        perm, oy, ox, flip = epoch_plan(n, gen)
        loss_sum = torch.zeros((), device=DEVICE)
        for s in range(steps):
            b = perm[s * batch_size:(s + 1) * batch_size]
            xb = augment(x_tr[b], oy[b], ox[b], flip[b])
            yb = y_tr[b]
            xb, y2, lam = mix_batch(xb, yb, mix_rng, MIX)
            out = model(xb)
            if y2 is None:
                loss = F.cross_entropy(out, yb)
            else:
                loss = lam * F.cross_entropy(out, yb) + (1.0 - lam) * F.cross_entropy(out, y2)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
            loss_sum += loss.detach()
        train_loss = loss_sum.item() / steps
        if not np.isfinite(train_loss):
            raise FloatingPointError(f"{prefix}non-finite training loss at epoch {epoch + 1}")
        val_loss = val_acc = float("nan")
        if val_idx is not None:
            logits = predict_logits(model, x_va, T["eval_batch_size"])
            val_loss = F.cross_entropy(logits, y_va).item()
            val_acc = (logits.argmax(dim=1) == y_va).float().mean().item()
        history.append((epoch + 1, train_loss, val_loss, val_acc))
        bar.set_postfix(loss=f"{train_loss:.3f}", val_acc=f"{val_acc:.3f}")
        if (epoch + 1) % T["print_every"] == 0 or epoch + 1 == epochs:
            tqdm.write(f"{prefix}ep {epoch + 1:3d}/{epochs} train_loss {train_loss:.4f} val_loss {val_loss:.4f} val_acc {val_acc:.4f} lr {sched.get_last_lr()[0]:.5f}")
    bar.close()
    return {
        "model": model,
        "mean": mean,
        "std": std,
        "history": history,
        "val_logits": None if logits is None else logits.float().cpu().numpy(),
        "params": count_params(model),
        "seconds": time.time() - t0,
    }

In [ ]:
# Cell 7: Experiment runner
BASE_AUG = dict(AUG0)
BASE_TRAIN = dict(TRAIN0)
EXPERIMENTS = CONFIG["experiments"]


def apply_experiment(name):
    exp = EXPERIMENTS[name]
    AUG.clear()
    AUG.update(BASE_AUG)
    AUG.update(exp.get("aug", {}))
    T.clear()
    T.update(BASE_TRAIN)
    T.update(exp.get("train", {}))
    globals()["MIX"] = AUG["mix"]
    if AUG["mix"] not in ("none", "cutout", "cutmix", "mixup"):
        raise ValueError(f"Unknown mix mode: {AUG['mix']}")
    if not 0.0 <= T["dropout"] < 1.0:
        raise ValueError("dropout must be in [0, 1)")

In [ ]:
# Cell 8: Final training of the 12 ensemble members on all training images
if "apply_experiment" not in globals():
    raise RuntimeError("Run Cells 0-7 in this kernel first (Cell 7 defines apply_experiment)")

FINAL_NAMES = ["DROP_EP480", "DROP_EP480_MIX", "DROP_EP480_CUTOUT", "DROP_EP480_CUTMIX"]
FINAL_DIR = Path(CONFIG["paths"]["results_dir"]) / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_PARAMS = CONFIG["expected_params"]["resnet_32_64_128"]

missing = [n for n in FINAL_NAMES if n not in CONFIG["experiments"]]
if missing:
    raise KeyError(f"Experiments missing from CONFIG: {missing}")


def make_ckpt(state, name, seed, spec, mean, std, params, loss, secs):
    return {
        "state_dict": state,
        "model_key": spec["model"],
        "member": name,
        "seed": seed,
        "mean": mean,
        "std": std,
        "params": params,
        "config": {"spec": dict(spec), "aug": dict(AUG), "train": dict(T)},
        "final_train_loss": float(loss),
        "seconds": float(secs),
        "train_images_sha256": str(DATA_RECORD["train_images_sha256"]),
        "torch_version": str(torch.__version__),
    }


def cpu_flip_logits(model, x, bs=256):
    outs = []
    with torch.no_grad():
        for i in range(0, len(x), bs):
            xb = x[i:i + bs]
            outs.append((model(xb) + model(xb.flip(3))) / 2)
    return torch.cat(outs)


all_idx = np.arange(len(Y_TRAIN))
plan, t_start = [], time.time()

for name in FINAL_NAMES:
    exp = CONFIG["experiments"][name]
    apply_experiment(name)
    spec = {**CONFIG["defaults"], **{k: v for k, v in exp.items() if k in CONFIG["defaults"]}}
    if spec["model"] != "resnet_32_64_128" or spec["epochs"] != 480 or float(T["dropout"]) != 0.2 or not T["flip_avg"]:
        raise ValueError(f"{name}: unexpected final settings (model {spec['model']}, epochs {spec['epochs']}, dropout {T['dropout']}, flip_avg {T['flip_avg']})")

    pre = FINAL_DIR / "_preflight.tmp"
    torch.save(make_ckpt(MODEL_BUILDERS[spec["model"]](NC).state_dict(), name, 0, spec, torch.zeros(3), torch.ones(3), EXPECTED_PARAMS, 0.0, 0.0), pre)
    torch.load(pre, map_location="cpu", weights_only=True)
    pre.unlink()

    for seed in spec["seeds"]:
        path = FINAL_DIR / f"final_{name}_s{seed}.pt"
        plan.append((name, seed, path))
        if path.exists():
            tqdm.write(f"SKIP {path.name}: exists (delete it deliberately to retrain)")
            continue

        out = train_one(spec["model"], spec["lr"], spec["wd"], spec["epochs"], spec["batch_size"], seed, all_idx, None, prefix=f"[final {name} s{seed}] ")
        model = out["model"]
        if out["params"] != EXPECTED_PARAMS:
            raise RuntimeError(f"{name} s{seed}: params {out['params']} != {EXPECTED_PARAMS}")

        tmp = path.with_name(path.name + ".tmp")
        torch.save(make_ckpt({k: v.detach().cpu() for k, v in model.state_dict().items()}, name, seed, spec,
                             out["mean"].cpu(), out["std"].cpu(), out["params"], out["history"][-1][1], out["seconds"]), tmp)
        os.replace(tmp, path)

        loaded = torch.load(path, map_location="cpu", weights_only=True)
        cpu_model = MODEL_BUILDERS[loaded["model_key"]](NC)
        cpu_model.load_state_dict(loaded["state_dict"])
        cpu_model.eval()
        x_cpu = (X_ALL.cpu().float() / 255.0 - loaded["mean"][None, :, None, None]) / loaded["std"][None, :, None, None]
        cpu_logits = cpu_flip_logits(cpu_model, x_cpu)
        gpu_logits = predict_logits(model, normalize(all_idx, out["mean"], out["std"]), 256).cpu()
        mismatch = float((cpu_logits.argmax(dim=1) != gpu_logits.argmax(dim=1)).float().mean())
        max_diff = float((cpu_logits - gpu_logits).abs().max())
        tqdm.write(f"saved {path.name}: params {out['params']}, final train loss {out['history'][-1][1]:.4f}, "
                   f"{out['seconds']:.0f} s | CPU vs GPU (flip-averaged) mismatch {mismatch:.5f}, max |logit diff| {max_diff:.2e}")
        if mismatch > CONFIG["final_check"]["parity_max_mismatch_frac"]:
            raise RuntimeError(f"{path.name}: CPU/GPU mismatch {mismatch:.5f} exceeds the configured limit")
        del out, model, cpu_model, loaded

if len(plan) != 12:
    raise RuntimeError(f"expected 12 members, plan has {len(plan)}")

manifest = []
ref_keys = list(MODEL_BUILDERS["resnet_32_64_128"](NC).state_dict().keys())
for name, seed, path in plan:
    ck = torch.load(path, map_location="cpu", weights_only=True)
    if ck["params"] != EXPECTED_PARAMS or list(ck["state_dict"].keys()) != ref_keys:
        raise RuntimeError(f"{path.name}: params or state_dict keys differ from a fresh resnet_32_64_128")
    if ck["train_images_sha256"] != str(DATA_RECORD["train_images_sha256"]):
        raise RuntimeError(f"{path.name}: trained on different training data")
    manifest.append({"file": path.name, "member": name, "seed": seed, "params": ck["params"],
                     "final_train_loss": ck["final_train_loss"], "seconds": ck["seconds"], "torch_version": ck["torch_version"]})
(FINAL_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"12 members present, total params {sum(m['params'] for m in manifest):,}; wall time this run {time.time() - t_start:.0f} s")
print("Cell 8 done")